In [3]:
import requests, jwt
from IPython.display import JSON

CATALOG_URL = "http://localhost:8181/catalog"
MANAGEMENT_URL = "http://localhost:8181/management"
KEYCLOAK_TOKEN_URL = "http://localhost:30080/realms/iceberg/protocol/openid-connect/token"

# Sign in

In [4]:
# Login to Keycloak
CLIENT_ID = "spark"
CLIENT_SECRET = "2OR3eRvYfSZzzZ16MlPd95jhLnOaLM52"

response = requests.post(
    url=KEYCLOAK_TOKEN_URL,
    data={
        "grant_type": "client_credentials",
        "client_id": CLIENT_ID,
        "client_secret": CLIENT_SECRET,
        "scope": "lakekeeper"
    },
    headers={"Content-type": "application/x-www-form-urlencoded"},
)
response.raise_for_status()
access_token = response.json()['access_token']

# Lets inspect the token we got to see that our application name is available:
JSON(jwt.decode(access_token, options={"verify_signature": False}))

<IPython.core.display.JSON object>

# Creating a Warehouse

In [5]:
response = requests.post(f"{MANAGEMENT_URL}/v1/warehouse",
              headers={
                    "Authorization": f"Bearer {access_token}"
                },
              json={
                # Name of the new warehouse
                "warehouse-name": "cepcan",
                # Physical location of this warehouse
                "storage-profile": {
                    "type": "s3",
                    "bucket": "iceberg",
                    "flavor": "minio", # For AWS Warhouses, use flavor "aws"
                    # you can change the prefix to something else, f. ex. f"{WAREHOUSE}
                    # as long as it is unique in the bucket
                    #"key-prefix": "path/to/new-warehouse/",
                    "assume-role-arn": None,
                    "endpoint": "http://localhost:9001",
                    #"sts-endpoint": "http://seaweedfs:8333",
                    #"sts-role-arn": "arn:aws:iam::000000000000:role/LakekeeperVendedRole",
                    "region": "us-east-1",
                    "path-style-access": True,
                    "sts-enabled": True
                },
                # Storage Credentials for the profile specified above.
                # These credentials are used to grant clients access to specific files in the storage.
                # Clients do not need to know those credentials and will never obtain them directly.
                "storage-credential": {
                    "type": "s3",
                    "credential-type": "access-key",
                    "access-key-id": "rustfsadmin",
                    "secret-access-key": "rustfsadmin"
                }
            })
print(f"{response.status_code}: {response.reason}")

201: Created
